# Лабораторная работа № 3
## kNN и Naive Bayes: фильтрация SMS-спама

**ФИО:** ...  
**Группа:** ...  
**Дата:** ...  

**Цель:** сравнить локальный метрический классификатор и вероятностный классификатор на одном реальном наборе данных.  
**Правило:** тестовая выборка используется только для финальной оценки. Все обучаемые преобразования находятся внутри `Pipeline`.


## План и оценивание

1. Данные и EDA — 10 баллов  
2. Ручные признаки и kNN — 30 баллов  
3. MultinomialNB + дополнительный NB — 20 баллов  
4. Оценка и интерпретация — 25 баллов  
5. Вывод и воспроизводимость — 15 баллов

Ячейки с `TODO` необходимо дополнить. После завершения выполните **Restart Kernel → Run All**.


In [ ]:
# При необходимости раскомментируйте:
# %pip install -q pandas numpy scikit-learn matplotlib seaborn

from io import BytesIO
from urllib.request import urlopen
from zipfile import ZipFile
import re, string, time, sys, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn import __version__ as sklearn_version
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     GridSearchCV, cross_val_predict)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB, ComplementNB, BernoulliNB
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, average_precision_score,
                             confusion_matrix, ConfusionMatrixDisplay,
                             precision_recall_curve)

RANDOM_STATE = 42
sns.set_theme(style='whitegrid')
print('Python:', sys.version.split()[0], '| pandas:', pd.__version__, '| sklearn:', sklearn_version)


## 1. Загрузка данных

Используется [UCI SMS Spam Collection](https://archive.ics.uci.edu/dataset/228/sms+spam+collection). Ожидается 5 574 англоязычных SMS с метками `ham` и `spam`.


In [ ]:
URL = 'https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip'
with ZipFile(BytesIO(urlopen(URL).read())) as z:
    with z.open('SMSSpamCollection') as f:
        df = pd.read_csv(f, sep='\t', header=None, names=['label', 'text'])

df['target'] = (df['label'] == 'spam').astype('int8')
assert set(df['label']) == {'ham', 'spam'}
assert df['text'].notna().all()
print(df.shape)
df.head()


### Задание 1. Контроль качества и EDA

- Найдите пропуски и полные дубликаты.
- Покажите число и долю объектов каждого класса.
- Сравните длину SMS в символах и словах по классам.
- Постройте минимум два графика.
- Под каждым графиком добавьте 2–3 предложения интерпретации.


In [ ]:
# TODO 1.1: качество данных и баланс классов
quality = pd.DataFrame({
    'value': [len(df), df.isna().sum().sum(), df.duplicated().sum()],
}, index=['rows', 'missing_cells', 'full_duplicates'])
display(quality)

class_stats = ...  # TODO: count и share для label
class_stats


In [ ]:
# TODO 1.2: длины и визуализации
df['n_chars'] = df['text'].str.len()
df['n_words'] = df['text'].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# TODO: countplot классов на axes[0]
# TODO: boxplot/histplot длины по классам на axes[1]
plt.tight_layout()
plt.show()


> **Вывод EDA:** TODO — опишите дисбаланс, различия по длине и минимум две гипотезы для feature engineering.


## 2. Train/test split

Тест — 20%, стратификация по target. Не удаляйте наблюдения после split на основании информации из test.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['target'], test_size=0.20,
    stratify=df['target'], random_state=RANDOM_STATE
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
print('train:', X_train.shape, 'spam share:', y_train.mean().round(4))
print('test :', X_test.shape, 'spam share:', y_test.mean().round(4))


## 3. Ручные признаки для kNN

Реализуйте не менее 6 признаков. Рекомендуемый набор: длина в символах/словах, доля цифр, доля заглавных, доля пунктуации, число `!`, наличие URL, наличие валюты/слов `free`, `win`, `prize`, `cash`.


In [ ]:
class SMSStats(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        rows = []
        for text in pd.Series(X).fillna('').astype(str):
            n = max(len(text), 1)
            letters = max(sum(ch.isalpha() for ch in text), 1)
            # TODO: вычислите минимум 6 осмысленных признаков
            features = [
                len(text),
                len(text.split()),
                # TODO ...
            ]
            rows.append(features)
        return np.asarray(rows, dtype=float)

# После реализации должно выполняться:
# assert SMSStats().fit_transform(X_train.iloc[:7]).shape[0] == 7


### Задание 2. Настройка kNN

Подберите `n_neighbors`, `weights`, `p` по F1 в 5-fold CV. Сравните `p=1` и `p=2`. Все этапы должны находиться в Pipeline.


In [ ]:
knn_pipe = Pipeline([
    ('stats', SMSStats()),
    ('scale', StandardScaler()),
    ('clf', KNeighborsClassifier())
])

knn_grid = {
    'clf__n_neighbors': [3, 5, 7, 11, 15, 21],
    'clf__weights': ['uniform', 'distance'],
    'clf__p': [1, 2]
}

# TODO: создайте GridSearchCV(scoring='f1', cv=cv, return_train_score=True)
knn_search = ...
# TODO: knn_search.fit(X_train, y_train)
# print(knn_search.best_params_, knn_search.best_score_)


In [ ]:
# TODO: график mean_test_score от k для комбинаций p × weights
# cv_knn = pd.DataFrame(knn_search.cv_results_)
# ...


> **Вывод по kNN:** TODO — укажите лучшие параметры, CV F1, роль scaling, признаки bias/variance и ограничения ручного представления.


## 4. Multinomial Naive Bayes

Текст преобразуется в неотрицательные счётчики токенов. Подберите `alpha`; по желанию настройте `min_df` и `ngram_range`.


In [ ]:
mnb_pipe = Pipeline([
    ('vec', CountVectorizer(lowercase=True)),
    ('clf', MultinomialNB())
])

mnb_grid = {
    'vec__ngram_range': [(1, 1), (1, 2)],
    'vec__min_df': [1, 2, 3],
    'clf__alpha': [0.01, 0.1, 0.3, 1.0, 3.0]
}

# TODO: GridSearchCV с теми же cv и scoring
mnb_search = ...
# TODO: fit и вывод best_params_, best_score_


## 5. Дополнительный Naive Bayes

Выберите **один** обязательный вариант:

- `ComplementNB` с TF–IDF или counts;
- `BernoulliNB` с `CountVectorizer(binary=True)`.

Сформулируйте, почему представление согласуется с моделью.


In [ ]:
# Вариант A: ComplementNB
cnb_pipe = Pipeline([
    ('vec', TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=2)),
    ('clf', ComplementNB())
])
cnb_grid = {'clf__alpha': [0.01, 0.1, 0.3, 1.0, 3.0]}

# Вариант B: BernoulliNB
bnb_pipe = Pipeline([
    ('vec', CountVectorizer(lowercase=True, binary=True, ngram_range=(1, 2), min_df=2)),
    ('clf', BernoulliNB())
])
bnb_grid = {'clf__alpha': [0.01, 0.1, 0.3, 1.0, 3.0]}

# TODO: выберите вариант, настройте через GridSearchCV и назовите nb2_search
nb2_search = ...


## 6. Финальная оценка на test

До этой секции не используйте test для выбора параметров. Оцените лучшие модели на одном и том же `X_test, y_test`.


In [ ]:
def evaluate_model(name, model, X, y):
    t0 = time.perf_counter()
    pred = model.predict(X)
    predict_ms = 1000 * (time.perf_counter() - t0)
    score = model.predict_proba(X)[:, 1]
    tn, fp, fn, tp = confusion_matrix(y, pred).ravel()
    return {
        'model': name,
        'accuracy': accuracy_score(y, pred),
        'precision': precision_score(y, pred, zero_division=0),
        'recall': recall_score(y, pred, zero_division=0),
        'f1': f1_score(y, pred, zero_division=0),
        'average_precision': average_precision_score(y, score),
        'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp,
        'predict_ms': predict_ms
    }

# TODO: заполните словарь обученными best_estimator_
models = {
    # 'kNN': knn_search.best_estimator_,
    # 'MultinomialNB': mnb_search.best_estimator_,
    # 'AdditionalNB': nb2_search.best_estimator_,
}

results = pd.DataFrame([
    evaluate_model(name, model, X_test, y_test)
    for name, model in models.items()
]).sort_values('f1', ascending=False) if models else pd.DataFrame()
results


In [ ]:
# TODO: confusion matrix для каждой модели
# fig, axes = plt.subplots(1, len(models), figsize=(5*len(models), 4))
# ...


In [ ]:
# TODO: Precision–Recall curves всех моделей на одном графике
# Не используйте ROC-AUC вместо PR-кривой как единственный график при дисбалансе.


> **Сравнение:** TODO — какая модель выигрывает по F1/AP? Какая минимизирует FP? Какая минимизирует FN? Совпадает ли лучший CV-кандидат с лучшим test-кандидатом?


## 7. Объяснение решений

### 7.1. Соседи kNN
Выберите 3–5 тестовых сообщений разных типов (TP/TN/FP/FN) и выведите их ближайших соседей из train с расстояниями и метками.


In [ ]:
# TODO: используйте шаги pipeline для преобразования и kneighbors
# best_knn = knn_search.best_estimator_
# Xtr_num = best_knn[:-1].transform(X_train)
# Xte_num = best_knn[:-1].transform(X_test)
# distances, indices = best_knn.named_steps['clf'].kneighbors(Xte_num[[...]] )


### 7.2. Характерные токены Naive Bayes

Для бинарной задачи рассчитайте разность `feature_log_prob_[1] - feature_log_prob_[0]`. Большие положительные значения — токены в пользу spam, отрицательные — в пользу ham.


In [ ]:
# TODO: извлеките vectorizer и classifier из лучшего MNB
# vec = mnb_search.best_estimator_.named_steps['vec']
# clf = mnb_search.best_estimator_.named_steps['clf']
# terms = vec.get_feature_names_out()
# log_ratio = clf.feature_log_prob_[1] - clf.feature_log_prob_[0]
# Выведите top-15 в обе стороны.


## 8. Анализ ошибок

Создайте таблицу минимум из 5 ошибок. Для каждой укажите тип (`FP`/`FN`), score, фрагмент SMS, вероятную причину и идею улучшения.


In [ ]:
# TODO: анализ ошибок лучшей модели
# best_name = ...
# best_model = models[best_name]
# pred = best_model.predict(X_test)
# score = best_model.predict_proba(X_test)[:, 1]
# errors = pd.DataFrame({...})
# display(errors.head(...))


## 9. Повышенный уровень: выбор порога по цене ошибок

Учебная стоимость: `Cost = 10·FP + FN`. Выберите порог **только по out-of-fold score обучающей выборки**, затем один раз оцените этот порог на test.


In [ ]:
# TODO: замените best_model на выбранную модель
# best_model = ...
# oof_score = cross_val_predict(best_model, X_train, y_train, cv=cv,
#                               method='predict_proba', n_jobs=-1)[:, 1]
# rows = []
# for threshold in np.arange(0.05, 0.96, 0.05):
#     pred = (oof_score >= threshold).astype(int)
#     tn, fp, fn, tp = confusion_matrix(y_train, pred).ravel()
#     rows.append({'threshold': threshold, 'fp': fp, 'fn': fn,
#                  'cost': 10*fp + fn})
# threshold_table = pd.DataFrame(rows)
# best_threshold = threshold_table.loc[threshold_table['cost'].idxmin(), 'threshold']
# threshold_table


## 10. Финальный инженерный вывод

Напишите 5–8 предложений:

1. Какая модель и конфигурация выбрана?
2. Каковы test precision, recall, F1, AP, FP и FN?
3. Почему выбранное представление подходит методу?
4. Какова цена основных ошибок?
5. Какой порог рекомендуется и как он был выбран?
6. Какие ограничения имеет эксперимент?
7. Что мониторить после внедрения: drift, долю spam, FP/FN по обратной связи, latency?

> **Вывод:** TODO


## 11. Самопроверка

- [ ] Notebook выполняется сверху вниз без ошибок.  
- [ ] Есть kNN, MultinomialNB и дополнительный NB.  
- [ ] Нет preprocessing до split.  
- [ ] Все параметры выбраны только на train.  
- [ ] Есть единая таблица test-метрик и матрицы ошибок.  
- [ ] Есть анализ минимум 5 ошибок.  
- [ ] Вывод содержит численные результаты и ограничения.


In [ ]:
# Версии окружения для воспроизводимости
pd.Series({
    'python': platform.python_version(),
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'scikit-learn': sklearn_version
}, name='version')
